# Invest UAE — Signal Detection Pipeline Demo

End-to-end demonstration of the AI-powered investment signal detection pipeline.

This notebook walks through each stage:
1. **Fetch** articles from 18+ MENA/global RSS sources
2. **Embed** and score relevance using sentence-transformers
3. **Classify** signal types (funding, expansion, partnership, etc.)
4. **Extract** entities (companies, locations, funding amounts)
5. **Score** investability and UAE alignment
6. **Rank** and output the final pipeline

In [ ]:
import sys
sys.path.insert(0, '..')

import asyncio
import json
from datetime import datetime, timezone

# Import our agents
from app.agents.embedding_agent import EmbeddingAgent
from app.agents.classifier_agent import ClassifierAgent
from app.agents.entity_agent import EntityAgent
from app.agents.scoring_agent import ScoringAgent

print('All agents imported successfully!')

## Stage 1: Initialize Agents

In [ ]:
# Initialize all four agents
embedding_agent = EmbeddingAgent()
classifier_agent = ClassifierAgent()
entity_agent = EntityAgent()
scoring_agent = ScoringAgent()

print('Embedding Agent:', 'ML mode' if not embedding_agent._fallback_mode else 'Fallback mode')
print('Classifier Agent: Ready')
print('Entity Agent: Ready')
print('Scoring Agent: Ready')

## Stage 2: Test with Sample Articles

Let's test the pipeline with realistic sample articles before running on live data.

In [ ]:
# Sample articles simulating real RSS feed data
SAMPLE_ARTICLES = [
    {
        'title': 'Tabby raises USD 200M Series D at USD 3.5B valuation',
        'text': 'UAE BNPL fintech Tabby has raised USD 200 million in its Series D round, '
                'led by Wellington Management. The Dubai-based company now has 10M+ active '
                'shoppers across the MENA region and plans to expand to Saudi Arabia.',
        'source': 'Wamda',
        'region': 'MENA',
    },
    {
        'title': 'Bain Capital opens Abu Dhabi office for MENA deals',
        'text': 'US private equity firm Bain Capital has opened a new office in Abu Dhabi, '
                'signaling commitment to regional capital deployment. The firm manages '
                'over USD 180B in assets globally.',
        'source': 'Arabian Business',
        'region': 'GCC',
    },
    {
        'title': 'G42 and Microsoft expand sovereign cloud to five markets',
        'text': 'Abu Dhabi AI company G42 has partnered with Microsoft to deploy sovereign '
                'AI cloud infrastructure across five Middle Eastern markets. The Abu Dhabi '
                'anchor hub will serve as the primary compute node.',
        'source': 'TechCrunch',
        'region': 'Global',
    },
    {
        'title': 'Stripe secures DIFC license, opens Dubai office',
        'text': 'Global payments company Stripe has received regulatory approval from DIFC '
                'to operate in the UAE. The San Francisco-based fintech will open a Dubai '
                'office to serve the MENA payments market.',
        'source': 'Khaleej Times',
        'region': 'UAE',
    },
    {
        'title': 'Manchester United signs new sponsorship deal',
        'text': 'Manchester United football club has signed a new shirt sponsorship deal '
                'worth GBP 60 million per year with a major sportswear brand.',
        'source': 'BBC Sport',
        'region': 'Global',
    },
]

print(f'Loaded {len(SAMPLE_ARTICLES)} sample articles')

## Stage 3: Relevance Scoring (Embedding Agent)

In [ ]:
print('=== Relevance Scores ===')
print(f'{"Article":<55} {"Score":>6} {"Relevant?":>10}')
print('-' * 75)

for article in SAMPLE_ARTICLES:
    text = f"{article['title']} {article['text']}"
    score = embedding_agent.relevance_score(text)
    relevant = 'YES' if score >= 0.2 else 'no'
    print(f'{article["title"][:53]:<55} {score:>5.2f} {relevant:>10}')

## Stage 4: Signal Classification (Classifier Agent)

In [ ]:
print('=== Signal Classification ===')
print(f'{"Article":<40} {"Type":<15} {"Strength":<10} {"Confidence":>10}')
print('-' * 80)

for article in SAMPLE_ARTICLES:
    text = f"{article['title']} {article['text']}"
    result = classifier_agent.classify(text)
    print(f'{article["title"][:38]:<40} {result.signal_type:<15} '
          f'{result.strength:<10} {result.confidence:>9.1%}')

## Stage 5: Entity Extraction (Entity Agent)

In [ ]:
print('=== Entity Extraction ===')
for article in SAMPLE_ARTICLES[:4]:  # Skip the irrelevant one
    entities = entity_agent.extract(article['title'], article['text'])
    print(f'\nArticle: {article["title"][:60]}')
    for e in entities[:2]:
        print(f'  Company: {e.name}')
        print(f'  HQ: {e.headquarters_city or "?"}, {e.headquarters_country or "?"}')
        if e.funding_usd:
            print(f'  Funding: ${e.funding_usd:,.0f}')
        if e.expansion_cities:
            print(f'  Expansion targets: {", ".join(e.expansion_cities)}')

## Stage 6: Scoring (Scoring Agent)

In [ ]:
print('=== Company Scoring ===')
print(f'{"Company":<25} {"Investability":>13} {"UAE Alignment":>13} {"Composite":>10}')
print('-' * 65)

test_companies = [
    {
        'name': 'Tabby',
        'signals': [{'type': 'funding', 'strength': 'high', 'headline': 'Series D raise', 'rationale': 'BNPL growth'}],
        'sectors': ['fintech', 'ecommerce'],
        'hq_country_code': 'AE',
        'expansion_target_codes': ['SA'],
        'funding_usd': 200_000_000,
        'description': 'UAE BNPL fintech with 10M+ shoppers',
    },
    {
        'name': 'Stripe',
        'signals': [{'type': 'expansion', 'strength': 'high', 'headline': 'DIFC license', 'rationale': 'UAE entry'}],
        'sectors': ['fintech'],
        'hq_country_code': 'US',
        'expansion_target_codes': ['AE'],
        'funding_usd': None,
        'description': 'Global payments infrastructure company',
    },
    {
        'name': 'G42',
        'signals': [
            {'type': 'partnership', 'strength': 'high', 'headline': 'Microsoft cloud deal', 'rationale': 'Sovereign AI'},
            {'type': 'funding', 'strength': 'high', 'headline': 'USD 1.5B from Microsoft', 'rationale': 'AI investment'},
        ],
        'sectors': ['artificial_intelligence', 'healthcare'],
        'hq_country_code': 'AE',
        'expansion_target_codes': ['US'],
        'funding_usd': 1_500_000_000,
        'description': 'Abu Dhabi AI and cloud computing company',
    },
]

for company in test_companies:
    scores = scoring_agent.score(**{k: v for k, v in company.items() if k != 'name'})
    print(f'{company["name"]:<25} {scores.investability_score:>12.1f} '
          f'{scores.uae_alignment_score:>12.1f} {scores.composite_score:>9.1f}')
    print(f'  Thesis: {scores.thesis_snippet}')

## Stage 7: Sector Detection (Embedding Agent)

In [ ]:
print('=== Sector Detection ===')
for article in SAMPLE_ARTICLES[:4]:
    text = f"{article['title']} {article['text']}"
    sectors = embedding_agent.top_sectors(text)
    print(f'{article["title"][:50]:<52} -> {", ".join(sectors)}')

## Summary

The pipeline successfully:
- Filters irrelevant articles (sports news scored low)
- Classifies investment signal types with confidence scores
- Extracts company entities, locations, and funding amounts
- Scores companies on investability and UAE alignment
- Detects relevant sectors using semantic embeddings

All of this runs **without any API keys** using open-source models.